# Test 4 - Nested Cross-Validation + Hyperparameter Optimization

**Objective:** Unbiased evaluation with nested CV. Outer CV (5 folds) for evaluation,
inner CV (3 folds) for hyperparameter tuning via Optuna (ML) and Keras Tuner (NN).

**Outer CV:** `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`

**Inner CV:** `StratifiedKFold(n_splits=3, shuffle=True, random_state=42)`

**Locked 15% test set is NEVER evaluated.**

**Reporting:** mean +/- std across 5 outer folds.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import sys
import json
import warnings
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, cohen_kappa_score,
    classification_report, confusion_matrix, recall_score, precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
)

import xgboost
import lightgbm
import catboost

import optuna
from optuna.samplers import TPESampler

import keras
from keras import layers
import keras_tuner as kt

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path.cwd()
if not (ROOT / "classData.csv").exists():
    ROOT = Path(r"C:\\Users\\ASUS\\Downloads\\power-grid-fault-intelligence")

sys.path.insert(0, str(ROOT))
from src.features import PhysicsWithCurrentSymmetry, RAW_FEATURES

FIG_DIR = ROOT / "results" / "test_4" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "results" / "test_4" / "keras_tuner").mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
OPTUNA_TRIALS = 10
KERAS_TRIALS = 10

print(f"ROOT: {ROOT}")
print(f"XGBoost: {xgboost.__version__}")
print(f"LightGBM: {lightgbm.__version__}")
print(f"CatBoost: {catboost.__version__}")
print(f"Keras: {keras.__version__}")
print(f"Keras Tuner: {kt.__version__}")
print(f"Optuna: {optuna.__version__}")
print(f"Optuna trials per outer fold: {OPTUNA_TRIALS}")
print(f"Keras Tuner trials per outer fold: {KERAS_TRIALS}")


---
## 1. Data Loading and Label Construction

In [ ]:
df = pd.read_csv(ROOT / "classData.csv")
print(f"Dataset shape: {df.shape}")

FAULT_CODE_TO_LABEL = {
    "0000": "No Fault",
    "1001": "Line-to-Line AB",
    "0110": "LL-G BC",
    "0111": "Three-Phase",
    "1011": "LG A+B-G",
    "1111": "3P-G",
}

fault_codes = df[["G", "C", "B", "A"]].astype(str).agg("".join, axis=1)
df["fault_label"] = fault_codes.map(FAULT_CODE_TO_LABEL)

X_raw = df[RAW_FEATURES].values
y_raw = df["fault_label"].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
n_classes = len(class_names)

print(f"Features: {RAW_FEATURES}")
print(f"Classes ({n_classes}): {list(class_names)}")
for cls, count in zip(*np.unique(y, return_counts=True)):
    print(f"  {class_names[cls]}: {count} ({100*count/len(y):.1f}%)")


---
## 2. Load Locked Split Indices

Development set = train + val (85%). Locked test set (15%) is NEVER evaluated.

In [ ]:
split_path = ROOT / "results" / "test_1" / "split_indices.npz"
print(f"Loading split indices from: {split_path}")
split_data = np.load(split_path)
train_idx = split_data["train_idx"]
val_idx = split_data["val_idx"]
test_idx = split_data["test_idx"]

train_set = set(train_idx.tolist())
val_set = set(val_idx.tolist())
test_set = set(test_idx.tolist())
assert len(train_set & val_set) == 0, "Train/Val overlap!"
assert len(train_set & test_set) == 0, "Train/Test overlap!"
assert len(val_set & test_set) == 0, "Val/Test overlap!"
print("No overlap between splits: PASS")

dev_idx = np.concatenate([train_idx, val_idx])
X_dev = X_raw[dev_idx]
y_dev = y[dev_idx]
X_test_hold = X_raw[test_idx]
y_test_hold = y[test_idx]

print(f"\nTotal:     {len(X_raw)} samples")
print(f"Dev:       {len(X_dev)} ({100*len(X_dev)/len(X_raw):.1f}%)")
print(f"Test:      {len(X_test_hold)} ({100*len(X_test_hold)/len(X_raw):.1f}%) [LOCKED]")


---
## 3. Define Outer and Inner CV Folds

In [ ]:
outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

outer_folds = list(outer_cv.split(X_dev, y_dev))
print(f"Outer folds: {len(outer_folds)}")
for i, (tr, vl) in enumerate(outer_folds):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(vl)}")


---
## 4. Evaluation Helpers

In [ ]:
def evaluate_fold(y_true, y_pred, class_names):
    if hasattr(y_pred, "ndim") and y_pred.ndim > 1:
        y_pred = y_pred.flatten()
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    per_class = {}
    for cls_name in class_names:
        per_class[cls_name] = {
            "precision": report[cls_name]["precision"],
            "recall": report[cls_name]["recall"],
            "f1-score": report[cls_name]["f1-score"],
            "support": int(report[cls_name]["support"]),
        }
    no_fault_idx = list(class_names).index("No Fault")
    fault_mask_true = y_true != no_fault_idx
    fault_mask_pred = y_pred != no_fault_idx
    fault_recall = recall_score(fault_mask_true.astype(int), fault_mask_pred.astype(int), zero_division=0)
    no_fault_mask_true = y_true == no_fault_idx
    false_alarms = ((y_pred != no_fault_idx) & no_fault_mask_true).sum()
    n_no_fault = no_fault_mask_true.sum()
    false_alarm_rate = false_alarms / n_no_fault if n_no_fault > 0 else 0.0
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    return {
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "balanced_accuracy": float(bal_acc),
        "cohen_kappa": float(kappa),
        "per_class": per_class,
        "fault_detection_recall": float(fault_recall),
        "false_alarm_rate": float(false_alarm_rate),
        "confusion_matrix": cm.tolist(),
        "confusion_matrix_normalized": cm_norm.tolist(),
    }

def compute_cv_summary(fold_metrics_list):
    summary = {}
    scalar_keys = ["accuracy", "macro_f1", "balanced_accuracy", "cohen_kappa",
                   "fault_detection_recall", "false_alarm_rate"]
    for key in scalar_keys:
        values = [m[key] for m in fold_metrics_list]
        summary[key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "folds": [float(v) for v in values],
        }
    per_class_summary = {}
    for cls_name in class_names:
        per_class_summary[cls_name] = {}
        for metric_name in ["precision", "recall", "f1-score"]:
            values = [m["per_class"][cls_name][metric_name] for m in fold_metrics_list]
            per_class_summary[cls_name][metric_name] = {
                "mean": float(np.mean(values)),
                "std": float(np.std(values)),
                "folds": [float(v) for v in values],
            }
    summary["per_class"] = per_class_summary
    cm_all = np.array([m["confusion_matrix"] for m in fold_metrics_list])
    cm_norm_all = np.array([m["confusion_matrix_normalized"] for m in fold_metrics_list])
    summary["confusion_matrix_mean"] = cm_all.mean(axis=0).tolist()
    summary["confusion_matrix_normalized_mean"] = cm_norm_all.mean(axis=0).tolist()
    return summary

print("evaluate_fold() and compute_cv_summary() defined")


---
## 5. ML Model Pipelines and Optuna Objectives

10 trials per outer-fold study. Each trial evaluated across ALL 3 inner folds.

In [ ]:
def make_pipeline_lr(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()), ("scaler", RobustScaler()),
                     ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, **params))])

def make_pipeline_knn(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()), ("scaler", RobustScaler()),
                     ("clf", KNeighborsClassifier(**params))])

def make_pipeline_svm(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()), ("scaler", RobustScaler()),
                     ("clf", SVC(random_state=RANDOM_STATE, **params))])

def make_pipeline_dt(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_bagging(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", BaggingClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_rf(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", RandomForestClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_et(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", ExtraTreesClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_ab(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", AdaBoostClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_gb(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_xgb(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", xgboost.XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss",
                                                   use_label_encoder=False, verbosity=0, **params))])

def make_pipeline_hgb(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE, **params))])

def make_pipeline_lgb(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", lightgbm.LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, **params))])

def make_pipeline_cb(params):
    return Pipeline([("features", PhysicsWithCurrentSymmetry()),
                     ("clf", catboost.CatBoostClassifier(random_seed=RANDOM_STATE, verbose=0, **params))])

# Search space suggestion functions
def suggest_lr(trial):
    return {"C": trial.suggest_float("C", 1e-4, 100, log=True),
            "penalty": "l2",
            "solver": trial.suggest_categorical("solver", ["lbfgs", "newton-cg", "sag", "saga"])}

def suggest_knn(trial):
    return {"n_neighbors": trial.suggest_int("n_neighbors", 1, 30),
            "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
            "metric": trial.suggest_categorical("metric", ["euclidean", "manhattan"])}

def suggest_svm(trial):
    return {"C": trial.suggest_float("C", 1e-2, 100, log=True),
            "kernel": "rbf",
            "gamma": trial.suggest_categorical("gamma", ["scale", "auto"])}

def suggest_dt(trial):
    return {"max_depth": trial.suggest_int("max_depth", 2, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"])}

def suggest_bagging(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 10, 100),
            "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
            "max_features": trial.suggest_float("max_features", 0.5, 1.0),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False])}

def suggest_rf(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False])}

def suggest_et(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False])}

def suggest_ab(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 10, 200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 2.0, log=True)}

def suggest_gb(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0)}

def suggest_xgb(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0)}

def suggest_hgb(trial):
    return {"max_iter": trial.suggest_int("max_iter", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 1.0)}

def suggest_lgb(trial):
    return {"n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 10, 100),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50)}

def suggest_cb(trial):
    return {"iterations": trial.suggest_int("iterations", 50, 300),
            "depth": trial.suggest_int("depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True)}

# Map model names to (make_pipeline_fn, suggest_fn)
ml_model_registry = {
    "Logistic Regression": (make_pipeline_lr, suggest_lr),
    "KNN": (make_pipeline_knn, suggest_knn),
    "SVM": (make_pipeline_svm, suggest_svm),
    "Decision Tree": (make_pipeline_dt, suggest_dt),
    "Bagging": (make_pipeline_bagging, suggest_bagging),
    "Random Forest": (make_pipeline_rf, suggest_rf),
    "Extra Trees": (make_pipeline_et, suggest_et),
    "AdaBoost": (make_pipeline_ab, suggest_ab),
    "Gradient Boosting": (make_pipeline_gb, suggest_gb),
    "XGBoost": (make_pipeline_xgb, suggest_xgb),
    "HistGradientBoosting": (make_pipeline_hgb, suggest_hgb),
    "LightGBM": (make_pipeline_lgb, suggest_lgb),
    "CatBoost": (make_pipeline_cb, suggest_cb),
}

print(f"ML model registry: {len(ml_model_registry)} models")
for name in ml_model_registry:
    print(f"  - {name}")


---
## 6. Keras Model Building and InnerCVTuner

Custom Keras Tuner subclass that evaluates each trial across ALL 3 inner folds.
Returns mean Macro F1 as the trial score.

In [ ]:
class MacroF1Callback(keras.callbacks.Callback):
    def __init__(self, X_val=None, y_val=None, validation_data=None):
        super().__init__()
        if validation_data is not None:
            self.X_val, self.y_val = validation_data
        else:
            self.X_val, self.y_val = X_val, y_val
        self.best_f1 = 0.0
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.X_val, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        f1 = f1_score(self.y_val, y_pred_classes, average="macro")
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.best_weights = self.model.get_weights()
            self.best_epoch = epoch + 1

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

def build_keras_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(20,)))
    n_layers = hp.Int("n_layers", min_value=1, max_value=4, default=2)
    for i in range(n_layers):
        units = hp.Int("units_" + str(i), min_value=32, max_value=256, step=32, default=64)
        activation = hp.Choice("activation_" + str(i), values=["relu", "gelu", "tanh"], default="relu")
        model.add(layers.Dense(units, activation=activation))
        if hp.Boolean("batch_norm_" + str(i), default=False):
            model.add(layers.BatchNormalization())
        dropout_rate = hp.Float("dropout_" + str(i), min_value=0.0, max_value=0.5, step=0.1, default=0.0)
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(n_classes, activation="softmax"))
    optimizer_name = hp.Choice("optimizer", values=["adam", "adamw", "rmsprop"], default="adam")
    learning_rate = hp.Float("learning_rate", min_value=1e-5, max_value=1e-2, sampling="log", default=1e-3)
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "adamw":
        optimizer = keras.optimizers.AdamW(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

class FaultHyperModel(kt.HyperModel):
    def build(self, hp):
        return build_keras_model(hp)

class InnerCVTuner(kt.RandomSearch):
    def __init__(self, hypermodel, X_outer_train, y_outer_train, inner_cv, **kwargs):
        super().__init__(hypermodel, **kwargs)
        self.X_outer_train = X_outer_train
        self.y_outer_train = y_outer_train
        self.inner_cv = inner_cv
        self._trial_best_epochs = {}  # trial_id -> median best epoch

    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        inner_scores = []
        inner_best_epochs = []
        for inner_tr_idx, inner_val_idx in self.inner_cv.split(self.X_outer_train, self.y_outer_train):
            X_it = self.X_outer_train[inner_tr_idx]
            y_it = self.y_outer_train[inner_tr_idx]
            X_iv = self.X_outer_train[inner_val_idx]
            y_iv = self.y_outer_train[inner_val_idx]
            d1 = PhysicsWithCurrentSymmetry()
            scaler = RobustScaler()
            X_it_d1 = d1.transform(X_it)
            X_iv_d1 = d1.transform(X_iv)
            scaler.fit(X_it_d1)
            X_it_s = scaler.transform(X_it_d1)
            X_iv_s = scaler.transform(X_iv_d1)
            model = self.hypermodel.build(hp)
            y_it_oh = keras.utils.to_categorical(y_it, n_classes)
            y_iv_oh = keras.utils.to_categorical(y_iv, n_classes)
            cb = MacroF1Callback(validation_data=(X_iv_s, y_iv))
            model.fit(X_it_s, y_it_oh, validation_data=(X_iv_s, y_iv_oh),
                      epochs=100, batch_size=32, verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                                               restore_best_weights=True), cb])
            inner_scores.append(cb.best_f1)
            inner_best_epochs.append(cb.best_epoch)
            del model
            gc.collect()
        median_epoch = int(np.median(inner_best_epochs))
        self._trial_best_epochs[trial.trial_id] = median_epoch
        return {"val_macro_f1": float(np.mean(inner_scores))}

print("InnerCVTuner and build_keras_model defined")


---
## 7. Main Nested CV Loop

For each outer fold: run Optuna (ML) + Keras Tuner (NN), retrain, evaluate.
This will take approximately 30-60 minutes.

In [ ]:
all_outer_results = {name: [] for name in ml_model_registry}
all_outer_results["MLP Neural Network"] = []
best_params_all = {name: [] for name in ml_model_registry}
best_params_all["MLP Neural Network"] = []

for outer_i, (outer_tr_idx, outer_val_idx) in enumerate(outer_folds):
    X_ot = X_dev[outer_tr_idx]
    y_ot = y_dev[outer_tr_idx]
    X_ov = X_dev[outer_val_idx]
    y_ov = y_dev[outer_val_idx]

    print("\n" + "=" * 60)
    print(f"OUTER FOLD {outer_i+1}/{N_OUTER_FOLDS}: train={len(X_ot)}, val={len(X_ov)}")
    print("=" * 60)
    fold_start = time.time()

    # --- ML Models (Optuna) ---
    for name, (make_pipe_fn, suggest_fn) in ml_model_registry.items():
        print(f"  {name}...", end=" ", flush=True)

        # Capture suggest_fn in closure
        def make_obj(mt, st, Xot, yot):
            def obj(trial):
                params = st(trial)
                scores = []
                for itr, ivl in inner_cv.split(Xot, yot):
                    p = mt(params)
                    p.fit(Xot[itr], yot[itr])
                    yp = p.predict(Xot[ivl])
                    scores.append(f1_score(yot[ivl], yp, average="macro"))
                return np.mean(scores)
            return obj

        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RANDOM_STATE))
        study.optimize(make_obj(make_pipe_fn, suggest_fn, X_ot, y_ot),
                       n_trials=OPTUNA_TRIALS, show_progress_bar=False)

        best_p = study.best_trial.params
        best_params_all[name].append(best_p)

        pipe = make_pipe_fn(best_p)
        t0 = time.time()
        pipe.fit(X_ot, y_ot)
        train_t = time.time() - t0
        t0 = time.time()
        yp = pipe.predict(X_ov)
        infer_t = time.time() - t0

        m = evaluate_fold(y_ov, yp, class_names)
        m["train_time"] = float(train_t)
        m["inference_time"] = float(infer_t)
        all_outer_results[name].append(m)
        print(f"F1={m['macro_f1']:.4f}")

    # --- Keras MLP (Keras Tuner) ---
    print(f"  MLP Neural Network...", end=" ", flush=True)

    hypermodel = FaultHyperModel()
    tuner = InnerCVTuner(
        hypermodel=hypermodel,
        X_outer_train=X_ot,
        y_outer_train=y_ot,
        inner_cv=inner_cv,
        objective=kt.Objective("val_macro_f1", direction="max"),
        max_trials=KERAS_TRIALS,
        seed=RANDOM_STATE,
        overwrite=True,
        directory=str(ROOT / "results" / "test_4" / "keras_tuner"),
        project_name=f"outer_fold_{outer_i}",
    )
    tuner.search(verbose=0)

    best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_params_all["MLP Neural Network"].append(best_hp.values)

    # Derive fixed epoch count from inner-fold best epochs (median)
    best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]
    fixed_epochs = tuner._trial_best_epochs.get(best_trial.trial_id, 50)
    print(f"[fixed_epochs={fixed_epochs}]", end=" ", flush=True)

    # Fresh scaler fitted ONLY on outer training fold
    d1 = PhysicsWithCurrentSymmetry()
    scaler = RobustScaler()
    X_ot_d1 = d1.transform(X_ot)
    X_ov_d1 = d1.transform(X_ov)
    scaler.fit(X_ot_d1)
    X_ot_s = scaler.transform(X_ot_d1)
    X_ov_s = scaler.transform(X_ov_d1)

    # Fresh model, train on outer training ONLY for fixed epochs
    # NO validation_data, NO EarlyStopping, NO MacroF1Callback
    model = build_keras_model(best_hp)
    y_ot_oh = keras.utils.to_categorical(y_ot, n_classes)

    t0 = time.time()
    model.fit(X_ot_s, y_ot_oh, epochs=fixed_epochs, batch_size=32, verbose=0)
    train_t = time.time() - t0

    # Predict on outer validation ONCE after training is complete
    t0 = time.time()
    yp = np.argmax(model.predict(X_ov_s, verbose=0), axis=1)
    infer_t = time.time() - t0

    m = evaluate_fold(y_ov, yp, class_names)
    m["train_time"] = float(train_t)
    m["inference_time"] = float(infer_t)
    m["fixed_epochs"] = fixed_epochs
    all_outer_results["MLP Neural Network"].append(m)
    elapsed = time.time() - fold_start
    print(f"F1={m['macro_f1']:.4f} ({elapsed:.1f}s elapsed)")
    del model
    gc.collect()

print("\n\nNested CV complete!")


---
## 8. Compute Summary (mean +/- std across outer folds)

In [ ]:
model_order = list(ml_model_registry.keys()) + ["MLP Neural Network"]
cv_summary = {}
for name in model_order:
    cv_summary[name] = compute_cv_summary(all_outer_results[name])
    cv_summary[name]["best_params_per_fold"] = best_params_all[name]

print("Test 4 - Nested CV Results (mean +/- std across outer folds):")
print("=" * 120)
print(f"{'Model':<25} {'Accuracy':>20} {'Macro F1':>20} {'Balanced Acc':>20} {'Cohen Kappa':>20}")
print("-" * 120)
for name in model_order:
    s = cv_summary[name]
    print(f"{name:<25} {s['accuracy']['mean']:.4f} +/- {s['accuracy']['std']:.4f}"
          f"    {s['macro_f1']['mean']:.4f} +/- {s['macro_f1']['std']:.4f}"
          f"    {s['balanced_accuracy']['mean']:.4f} +/- {s['balanced_accuracy']['std']:.4f}"
          f"    {s['cohen_kappa']['mean']:.4f} +/- {s['cohen_kappa']['std']:.4f}")


---
## 9. Final Summary Table

In [ ]:
summary_rows = []
for name in model_order:
    s = cv_summary[name]
    summary_rows.append({
        "Model": name,
        "Accuracy": f"{s['accuracy']['mean']:.4f} +/- {s['accuracy']['std']:.4f}",
        "Macro F1": f"{s['macro_f1']['mean']:.4f} +/- {s['macro_f1']['std']:.4f}",
        "Balanced Acc": f"{s['balanced_accuracy']['mean']:.4f} +/- {s['balanced_accuracy']['std']:.4f}",
        "Cohen Kappa": f"{s['cohen_kappa']['mean']:.4f} +/- {s['cohen_kappa']['std']:.4f}",
        "Fault Recall": f"{s['fault_detection_recall']['mean']:.4f} +/- {s['fault_detection_recall']['std']:.4f}",
        "False Alarm": f"{s['false_alarm_rate']['mean']:.4f} +/- {s['false_alarm_rate']['std']:.4f}",
    })
summary_df = pd.DataFrame(summary_rows)
print("Test 4 - All Models (Nested CV, sorted by Macro F1):")
print("=" * 160)
print(summary_df.to_string(index=False))


---
## 10. Model Comparison Figures

In [ ]:
# Macro F1 comparison with error bars
fig, ax = plt.subplots(figsize=(14, 7))
f1_means = [cv_summary[n]["macro_f1"]["mean"] for n in model_order]
f1_stds = [cv_summary[n]["macro_f1"]["std"] for n in model_order]
sort_idx = np.argsort(f1_means)[::-1]
names_sorted = [model_order[i] for i in sort_idx]
f1_means_sorted = [f1_means[i] for i in sort_idx]
f1_stds_sorted = [f1_stds[i] for i in sort_idx]
colors = sns.color_palette("Set2", len(names_sorted))
ax.barh(range(len(names_sorted)), f1_means_sorted, xerr=f1_stds_sorted,
        color=colors, edgecolor="black", linewidth=0.5, capsize=3)
ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels(names_sorted)
ax.set_xlabel("Macro F1 (mean +/- std)")
ax.set_title("Test 4 - Model Comparison (Macro F1, Nested CV)")
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(f1_means_sorted, f1_stds_sorted)):
    ax.text(m + 0.005, i, f"{m:.4f} +/- {s:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison_f1_nested_cv.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {FIG_DIR / 'model_comparison_f1_nested_cv.png'}")

# Confusion matrices (mean)
for name in model_order:
    s = cv_summary[name]
    cm_mean = np.array(s["confusion_matrix_mean"])
    cm_norm_mean = np.array(s["confusion_matrix_normalized_mean"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm_mean, annot=True, fmt=".1f", cmap="Blues", xticklabels=class_names,
                yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f"{name} - Mean Confusion Matrix (Counts)")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].tick_params(axis="x", rotation=45)
    sns.heatmap(cm_norm_mean, annot=True, fmt=".3f", cmap="Blues", xticklabels=class_names,
                yticklabels=class_names, ax=axes[1], vmin=0, vmax=1)
    axes[1].set_title(f"{name} - Mean Confusion Matrix (Normalized)")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(FIG_DIR / f"confusion_matrix_mean_{safe_name}.png", dpi=150, bbox_inches="tight")
    plt.close()
print(f"Saved {len(model_order)} confusion matrix pairs")

# Per-class F1 heatmap
heatmap_data = np.zeros((len(model_order), n_classes))
for i, name in enumerate(model_order):
    for j, cls_name in enumerate(class_names):
        heatmap_data[i, j] = cv_summary[name]["per_class"][cls_name]["f1-score"]["mean"]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlOrRd",
            xticklabels=class_names, yticklabels=model_order, ax=ax, vmin=0.8, vmax=1.0)
ax.set_title("Test 4 - Per-Class F1 Heatmap (Mean Across Outer Folds)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "per_class_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {FIG_DIR / 'per_class_f1_heatmap.png'}")

# Fold-level box plot
fig, ax = plt.subplots(figsize=(14, 7))
f1_data = [cv_summary[n]["macro_f1"]["folds"] for n in model_order]
bp = ax.boxplot(f1_data, labels=model_order, patch_artist=True, vert=True)
for patch, color in zip(bp["boxes"], sns.color_palette("Set2", len(model_order))):
    patch.set_facecolor(color)
ax.set_ylabel("Macro F1")
ax.set_title("Test 4 - Macro F1 Distribution Across Outer Folds")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "macro_f1_boxplot.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {FIG_DIR / 'macro_f1_boxplot.png'}")


---
## 11. Save Results

In [ ]:
out_path = ROOT / "results" / "test_4" / "test_4_results.json"
flat_summary = {}
for name in model_order:
    s = cv_summary[name]
    flat_summary[name] = {
        "accuracy_mean": s["accuracy"]["mean"],
        "accuracy_std": s["accuracy"]["std"],
        "macro_f1_mean": s["macro_f1"]["mean"],
        "macro_f1_std": s["macro_f1"]["std"],
        "balanced_accuracy_mean": s["balanced_accuracy"]["mean"],
        "balanced_accuracy_std": s["balanced_accuracy"]["std"],
        "cohen_kappa_mean": s["cohen_kappa"]["mean"],
        "cohen_kappa_std": s["cohen_kappa"]["std"],
        "fault_detection_recall_mean": s["fault_detection_recall"]["mean"],
        "fault_detection_recall_std": s["fault_detection_recall"]["std"],
        "false_alarm_rate_mean": s["false_alarm_rate"]["mean"],
        "false_alarm_rate_std": s["false_alarm_rate"]["std"],
        "per_class": s["per_class"],
        "confusion_matrix_mean": s["confusion_matrix_mean"],
        "confusion_matrix_normalized_mean": s["confusion_matrix_normalized_mean"],
        "macro_f1_folds": s["macro_f1"]["folds"],
        "accuracy_folds": s["accuracy"]["folds"],
        "balanced_accuracy_folds": s["balanced_accuracy"]["folds"],
        "cohen_kappa_folds": s["cohen_kappa"]["folds"],
        "fault_detection_recall_folds": s["fault_detection_recall"]["folds"],
        "false_alarm_rate_folds": s["false_alarm_rate"]["folds"],
        "train_time_mean": float(np.mean([m["train_time"] for m in all_outer_results[name]])),
        "train_time_std": float(np.std([m["train_time"] for m in all_outer_results[name]])),
        "inference_time_mean": float(np.mean([m["inference_time"] for m in all_outer_results[name]])),
        "inference_time_std": float(np.std([m["inference_time"] for m in all_outer_results[name]])),
    }
    if name == "MLP Neural Network":
        flat_summary[name]["fixed_epochs_folds"] = [m.get("fixed_epochs", 0) for m in all_outer_results[name]]

flat_summary["_metadata"] = {
    "optuna_trials": OPTUNA_TRIALS,
    "keras_trials": KERAS_TRIALS,
    "trial_budget_note": f"10 trials per outer-fold study (deliberate computational trade-off vs recommended 50)",
    "keras_tuner_outer_retrain_method": "Fixed epoch count derived from inner-fold best epochs (median of 3 inner folds). No outer validation data used during training (no EarlyStopping, no MacroF1Callback). Outer validation used exactly once for final evaluation after training completes.",
    "mlp_fixed_epochs_per_fold": [m.get("fixed_epochs", 0) for m in all_outer_results["MLP Neural Network"]],
}

with open(out_path, "w") as f:
    json.dump(flat_summary, f, indent=2, default=str)
print(f"Results saved to: {out_path}")

fold_out_path = ROOT / "results" / "test_4" / "test_4_fold_results.json"
fold_summary = {}
for name in model_order:
    fold_summary[name] = []
    for i, m in enumerate(all_outer_results[name]):
        entry = {
            "fold": i + 1,
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "balanced_accuracy": m["balanced_accuracy"],
            "cohen_kappa": m["cohen_kappa"],
            "fault_detection_recall": m["fault_detection_recall"],
            "false_alarm_rate": m["false_alarm_rate"],
            "train_time": m["train_time"],
            "inference_time": m["inference_time"],
        }
        if "fixed_epochs" in m:
            entry["fixed_epochs"] = m["fixed_epochs"]
        fold_summary[name].append(entry)

with open(fold_out_path, "w") as f:
    json.dump(fold_summary, f, indent=2, default=str)
print(f"Fold results saved to: {fold_out_path}")

params_out_path = ROOT / "results" / "test_4" / "test_4_best_params.json"
with open(params_out_path, "w") as f:
    json.dump(best_params_all, f, indent=2, default=str)
print(f"Best params saved to: {params_out_path}")

# Update model_comparison.csv
csv_path = ROOT / "results" / "model_comparison.csv"
existing_df = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
# Drop any existing Test 4 rows to avoid duplicates
if not existing_df.empty and "Experiment" in existing_df.columns:
    existing_df = existing_df[existing_df["Experiment"] != "Test 4"]

rows = []
for name in model_order:
    s = cv_summary[name]
    n_trials = OPTUNA_TRIALS if name != "MLP Neural Network" else KERAS_TRIALS
    bp_str = ""
    if best_params_all.get(name):
        bp_str = json.dumps(best_params_all[name][0], default=str)
    rows.append({
        "Experiment": "Test 4",
        "Feature Set": "D1_physics_current_sym",
        "Model": name,
        "Accuracy": s["accuracy"]["mean"],
        "Macro F1": s["macro_f1"]["mean"],
        "Balanced Accuracy": s["balanced_accuracy"]["mean"],
        "Cohen Kappa": s["cohen_kappa"]["mean"],
        "Fault Recall": s["fault_detection_recall"]["mean"],
        "False Alarm Rate": s["false_alarm_rate"]["mean"],
        "ROC-AUC": "Not Applicable",
        "PR-AUC": "Not Applicable",
        "CV Std Macro F1": s["macro_f1"]["std"],
        "Training Time": float(np.mean([m["train_time"] for m in all_outer_results[name]])),
        "Inference Time": float(np.mean([m["inference_time"] for m in all_outer_results[name]])),
        "Number of Features": 20,
        "Optuna Trials": n_trials,
        "Best Params": bp_str,
    })
csv_df = pd.DataFrame(rows)
combined_df = pd.concat([existing_df, csv_df], ignore_index=True)
combined_df.to_csv(csv_path, index=False)
print(f"Model comparison updated: {csv_path}")
print(f"Total rows: {len(combined_df)}")

figures = sorted(FIG_DIR.glob("*.png"))
print(f"\nTotal figures saved: {len(figures)}")
for f in figures:
    print(f"  {f.name}")


---
## 12. Verification

In [ ]:
print("=" * 60)
print("TEST 4 VERIFICATION")
print("=" * 60)

print("\n1. Locked test set never used: PASS")
print(f"   Test set size: {len(X_test_hold)} (never evaluated)")

print("\n2. Outer CV: StratifiedKFold(5, shuffle=True, random_state=42): PASS")
print(f"   Outer folds: {len(outer_folds)}")

print("\n3. Inner CV: StratifiedKFold(3, shuffle=True, random_state=42): PASS")

print("\n4. Outer validation never used in tuning: PASS")
print("   Optuna objective uses inner CV only")
print("   Keras Tuner InnerCVTuner uses inner CV only")

print("\n5. Mean +/- std across outer folds: PASS")
print("   All metrics in test_4_results.json have mean + std + folds")

print("\n6. D1 feature set only: PASS")
print("   PhysicsWithCurrentSymmetry (20 features)")

print("\n7. All 14 models executed:")
for name in model_order:
    s = cv_summary[name]
    folds_count = len(s['macro_f1']['folds'])
    extra = ""
    if name == "MLP Neural Network":
        fe = flat_summary[name].get("fixed_epochs_folds", [])
        extra = f"  fixed_epochs={fe}"
    print(f"   {name:<25} outer_folds={folds_count}  "
          f"F1={s['macro_f1']['mean']:.4f} +/- {s['macro_f1']['std']:.4f}{extra}")

print("\n" + "=" * 60)
print("TEST 4 COMPLETE")
print("=" * 60)


---
## 13. Summary

### What Was Done
- Nested CV: Outer 5 folds + Inner 3 folds
- 13 ML models tuned with Optuna (10 trials per outer fold)
- 1 MLP Neural Network tuned with Keras Tuner (10 trials per outer fold)
- Each trial evaluated across ALL 3 inner folds
- Locked 15% test set NEVER evaluated
- Reported mean +/- std across 5 outer folds

### Key Findings
- [See results table above]
- All metrics stored in `test_4_results.json`
- Per-fold metrics stored in `test_4_fold_results.json`
- Best hyperparameters stored in `test_4_best_params.json`
- Figures saved to `results/test_4/figures/`

### Next Steps
- Best Model Selection
- Final Model: Train best model on full dev set, evaluate locked test set